# 05 — ML Modelleri & MLflow

**Hedef:** `delta/gold/ml_features` tablosundan 5 sınıflandırıcı eğitmek, MLflow'a loglamak ve tüm sonuçları görselleştirmek.

**Problem:** Binary classification — `label=1` (tutuklama yapıldı), `label=0` (yapılmadı)

**Modeller:** Logistic Regression · Decision Tree · Random Forest · GBT · Naive Bayes

**Temel iyileştirme:** Veri dengesizliği (~%85 tutuksuz / ~%15 tutuklu) → class-weight balancing ile recall artırımı

In [ ]:
import os
from pathlib import Path

# Delta JARs
_delta_pkg = 'io.delta:delta-spark_2.13:4.0.1'
os.environ['PYSPARK_SUBMIT_ARGS'] = f'--packages {_delta_pkg} pyspark-shell'

def _repo_root():
    from pathlib import Path
    cwd = Path.cwd().resolve()
    if (cwd / 'data' / 'raw').is_dir(): return cwd
    if cwd.name == 'notebooks' and (cwd.parent / 'data' / 'raw').is_dir(): return cwd.parent
    raise FileNotFoundError('Cannot find repo root')

REPO_ROOT    = _repo_root()
FEATURE_PATH = str(REPO_ROOT / 'delta' / 'gold' / 'ml_features')
REPORTS_DIR  = REPO_ROOT / 'reports'
FIGURES_DIR  = REPO_ROOT / 'dashboard' / 'figures' / 'ml'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MLFLOW_URI   = f"file:///{REPO_ROOT}/mlruns"

print('FEATURE_PATH:', FEATURE_PATH)
print('FIGURES_DIR :', FIGURES_DIR)
print('MLFLOW_URI  :', MLFLOW_URI)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display, Markdown

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lit
from pyspark.ml import Pipeline
from pyspark.ml.classification import (
    LogisticRegression, DecisionTreeClassifier,
    RandomForestClassifier, GBTClassifier, NaiveBayes,
)
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator, BinaryClassificationEvaluator,
)
from pyspark.ml.feature import StringIndexer, VectorAssembler

import mlflow
import mlflow.spark

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

def savefig(name):
    path = str(FIGURES_DIR / name)
    plt.savefig(path, bbox_inches='tight')
    print(f'Saved: {path}')

## 1. SparkSession

In [ ]:
_ex = SparkSession.getActiveSession()
if _ex: _ex.stop()

try:
    from delta import configure_spark_with_delta_pip
    spark = configure_spark_with_delta_pip(
        SparkSession.builder
        .appName('ML_Models')
        .master('local[*]')
        .config('spark.driver.memory', '4g')
        .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
        .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    ).getOrCreate()
except Exception:
    spark = (
        SparkSession.builder
        .appName('ML_Models')
        .master('local[*]')
        .config('spark.driver.memory', '4g')
        .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
        .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
        .getOrCreate()
    )

spark.sparkContext.setLogLevel('WARN')
print('Spark', spark.version)

## 2. Veri Yükleme & Sınıf Dengesi

In [ ]:
df = spark.read.format('delta').load(FEATURE_PATH).dropna(subset=['label'])

total = df.count()
pos   = df.filter(col('label') == 1.0).count()
neg   = total - pos

print(f'Total rows    : {total:,}')
print(f'Arrested  (1) : {pos:,}  ({100*pos/total:.1f}%)')
print(f'Not arr.  (0) : {neg:,}  ({100*neg/total:.1f}%)')

# Class weight balancing
weight_pos = neg / total
weight_neg = pos / total
df = df.withColumn('classWeight',
    when(col('label') == 1.0, lit(float(weight_pos))).otherwise(lit(float(weight_neg))))

# Visualise class imbalance
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['Not Arrested (0)', 'Arrested (1)'], [neg, pos], color=['#4C72B0', '#DD8452'])
ax.set_title('Label Distribution (class imbalance)')
ax.set_ylabel('Row count')
for i, v in enumerate([neg, pos]):
    ax.text(i, v + 200, f'{v:,}\n({100*v/total:.1f}%)', ha='center', fontsize=10)
plt.tight_layout()
savefig('label_distribution.png')
plt.show()

## 3. Train/Test Split & Feature Pipeline

In [ ]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
print(f'Train: {train_df.count():,}  |  Test: {test_df.count():,}')

categorical_cols = ['primary_type_group', 'location_group', 'district_group']
numeric_cols     = [
    'hour', 'day_of_week', 'month',
    'is_weekend', 'is_night',
    'domestic_numeric', 'lat_grid', 'lon_grid_abs',
]

indexers     = [StringIndexer(inputCol=c, outputCol=f'{c}_idx', handleInvalid='keep')
                for c in categorical_cols]
indexed_cols = [f'{c}_idx' for c in categorical_cols]
feature_cols = numeric_cols + indexed_cols
assembler    = VectorAssembler(inputCols=feature_cols, outputCol='features', handleInvalid='keep')

print('Features:', feature_cols)

## 4. Model Eğitimi & MLflow Loglama

5 model eğitilir. Her model için:
- Parametreler, metrikler ve model artifact MLflow'a loglanır
- `classWeight` sütunu class imbalance'ı düzeltmek için kullanılır (NaiveBayes hariç)

In [ ]:
models = {
    'LogisticRegression': LogisticRegression(
        featuresCol='features', labelCol='label', weightCol='classWeight',
        maxIter=100, regParam=0.01, elasticNetParam=0.1,
    ),
    'DecisionTreeClassifier': DecisionTreeClassifier(
        featuresCol='features', labelCol='label', weightCol='classWeight',
        maxDepth=10, seed=42,
    ),
    'RandomForestClassifier': RandomForestClassifier(
        featuresCol='features', labelCol='label', weightCol='classWeight',
        numTrees=100, maxDepth=10, seed=42,
    ),
    'GBTClassifier': GBTClassifier(
        featuresCol='features', labelCol='label',
        maxIter=100, maxDepth=6, stepSize=0.05, seed=42,
    ),
    'NaiveBayes': NaiveBayes(
        featuresCol='features', labelCol='label', smoothing=1.0, modelType='multinomial',
    ),
}

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment('chicago_crime_arrest_classification')

all_metrics      = []
all_predictions  = {}
best_model_name  = None
best_auc         = -1.0
best_predictions = None
best_pipeline    = None

for model_name, model in models.items():
    print(f'Training {model_name}...')
    pipeline       = Pipeline(stages=indexers + [assembler, model])

    with mlflow.start_run(run_name=model_name):
        pipe_model  = pipeline.fit(train_df)
        preds       = pipe_model.transform(test_df)

        # Metrics
        m = {}
        for key, mn in [('accuracy','accuracy'),('f1','f1'),
                         ('precision','weightedPrecision'),('recall','weightedRecall')]:
            m[key] = MulticlassClassificationEvaluator(
                labelCol='label', predictionCol='prediction', metricName=mn).evaluate(preds)
        m['auc_roc'] = BinaryClassificationEvaluator(
            labelCol='label', rawPredictionCol='rawPrediction',
            metricName='areaUnderROC').evaluate(preds)
        tp = preds.filter((col('label')==1.0)&(col('prediction')==1.0)).count()
        fn = preds.filter((col('label')==1.0)&(col('prediction')==0.0)).count()
        m['recall_arrested'] = tp/(tp+fn) if (tp+fn)>0 else 0.0

        mlflow.log_param('model_name', model_name)
        mlflow.log_param('class_balanced', True)
        mlflow.log_param('feature_count', len(feature_cols))
        for k, v in m.items():
            mlflow.log_metric(k, v)
        mlflow.spark.log_model(pipe_model, artifact_path=f'{model_name}_spark_model')

        row = {'model': model_name}; row.update(m)
        all_metrics.append(row)
        all_predictions[model_name] = preds

        print(f'  accuracy={m["accuracy"]:.3f}  auc={m["auc_roc"]:.3f}  recall_arrested={m["recall_arrested"]:.3f}')

        if m['auc_roc'] > best_auc:
            best_auc        = m['auc_roc']
            best_model_name = model_name
            best_predictions= preds
            best_pipeline   = pipe_model

metrics_df = pd.DataFrame(all_metrics)
print(f'\nBest model: {best_model_name}  AUC={best_auc:.4f}')
display(metrics_df.set_index('model').round(4))

## 5. Model Karşılaştırma Grafikleri

In [ ]:
# --- Grouped bar chart: 5 models × 5 metrics ---
metric_cols   = ['accuracy', 'f1', 'precision', 'recall', 'auc_roc']
metric_labels = ['Accuracy', 'F1', 'Precision', 'Recall', 'AUC-ROC']
model_names   = metrics_df['model'].tolist()
short_names   = [m.replace('Classifier','').replace('Regression','\nReg') for m in model_names]

x     = np.arange(len(model_names))
width = 0.15
colors= ['#2E86AB','#E84855','#3BB273','#F4A261','#9B59B6']

fig, ax = plt.subplots(figsize=(14, 6))
for i, (col_name, label) in enumerate(zip(metric_cols, metric_labels)):
    vals = metrics_df[col_name].values
    bars = ax.bar(x + i*width, vals, width, label=label, color=colors[i], alpha=0.88)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{v:.2f}', ha='center', va='bottom', fontsize=7)

ax.set_xticks(x + width*2)
ax.set_xticklabels(short_names, fontsize=10)
ax.set_ylim(0, 1.08)
ax.set_ylabel('Score')
ax.set_title('5 Model Performance Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
plt.tight_layout()
savefig('model_comparison.png')
plt.show()

In [ ]:
# --- AUC-ROC bar chart ---
mdf_sorted = metrics_df.sort_values('auc_roc', ascending=True)
palette    = ['#c0392b' if n == best_model_name else '#3498db'
              for n in mdf_sorted['model']]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(mdf_sorted['model'], mdf_sorted['auc_roc'], color=palette)
for bar, v in zip(bars, mdf_sorted['auc_roc']):
    ax.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
            f'{v:.4f}', va='center', fontsize=10)
ax.set_xlim(0, 1.05)
ax.set_xlabel('AUC-ROC')
ax.set_title('AUC-ROC by Model  (red = best)', fontweight='bold')
ax.axvline(0.5, color='gray', linestyle='--', linewidth=0.8)
plt.tight_layout()
savefig('auc_roc_comparison.png')
plt.show()

In [ ]:
# --- Recall for arrested class (minority) ---
fig, ax = plt.subplots(figsize=(9, 4))
colors2 = ['#e74c3c' if n == best_model_name else '#2ecc71' for n in metrics_df['model']]
bars = ax.bar(metrics_df['model'], metrics_df['recall_arrested'], color=colors2, alpha=0.85)
for bar, v in zip(bars, metrics_df['recall_arrested']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{v:.2f}', ha='center', fontsize=10)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Recall (arrested class)')
ax.set_title('Recall for Arrested Class (label=1) — higher is better', fontweight='bold')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
savefig('recall_arrested_comparison.png')
plt.show()

## 6. Feature Importance

In [ ]:
import csv

# Extract from best pipeline
stage = best_pipeline.stages[-1]
if hasattr(stage, 'featureImportances'):
    vals = stage.featureImportances.toArray()
elif hasattr(stage, 'coefficients'):
    vals = np.abs(stage.coefficients.toArray())
else:
    vals = np.zeros(len(feature_cols))

imp_df = pd.DataFrame({'feature': feature_cols, 'importance': vals})\
           .sort_values('importance', ascending=False)

# Save CSV
imp_csv = REPORTS_DIR / 'feature_importance_best_model.csv'
imp_df.to_csv(imp_csv, index=False)
print('Feature importance saved to:', imp_csv)

# Plot
fig, ax = plt.subplots(figsize=(9, 5))
colors3 = ['#e74c3c' if i == 0 else '#3498db' for i in range(len(imp_df))]
ax.barh(imp_df['feature'][::-1], imp_df['importance'][::-1], color=colors3[::-1])
ax.set_xlabel('Importance')
ax.set_title(f'Feature Importance — {best_model_name}', fontweight='bold')
for i, (feat, val) in enumerate(zip(imp_df['feature'][::-1], imp_df['importance'][::-1])):
    ax.text(val+0.002, i, f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout()
savefig('feature_importance.png')
plt.show()

## 7. Confusion Matrix (Best Model)

In [ ]:
cm_rows = (
    best_predictions
    .groupBy('label', 'prediction').count()
    .orderBy('label', 'prediction').collect()
)

# Build 2×2 matrix
cm = np.zeros((2, 2), dtype=int)
for r in cm_rows:
    cm[int(r['label'])][int(r['prediction'])] = r['count']

# Save CSV
cm_csv = REPORTS_DIR / 'confusion_matrix_best_model.csv'
pd.DataFrame([
    {'label': float(r['label']), 'prediction': float(r['prediction']), 'count': r['count']}
    for r in cm_rows
]).to_csv(cm_csv, index=False)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', ax=ax,
    xticklabels=['Pred: Not Arrested', 'Pred: Arrested'],
    yticklabels=['True: Not Arrested', 'True: Arrested'],
    linewidths=0.5
)
ax.set_title(f'Confusion Matrix — {best_model_name}', fontweight='bold')

# Annotate TN/FP/FN/TP
for (r, c), label in [((0,0),'TN'), ((0,1),'FP'), ((1,0),'FN'), ((1,1),'TP')]:
    ax.text(c+0.5, r+0.75, label, ha='center', color='gray', fontsize=11, fontweight='bold')

plt.tight_layout()
savefig('confusion_matrix.png')
plt.show()

tn, fp, fn, tp = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
print(f'TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}')
print(f'Precision (arrested) : {tp/(tp+fp):.3f}')
print(f'Recall    (arrested) : {tp/(tp+fn):.3f}')

## 8. ROC Curve

In [ ]:
from pyspark.ml.functions import vector_to_array

fig, ax = plt.subplots(figsize=(7, 6))
roc_colors = ['#2E86AB','#E84855','#3BB273','#F4A261','#9B59B6']

for i, (mname, preds) in enumerate(all_predictions.items()):
    auc = next(r['auc_roc'] for r in all_metrics if r['model'] == mname)

    # Get probability scores for positive class
    try:
        score_col = (
            preds
            .select(
                col('label'),
                vector_to_array(col('probability')).getItem(1).alias('score')
            )
            .toPandas()
            .sort_values('score', ascending=False)
        )

        thresholds = np.linspace(0, 1, 100)
        tprs, fprs = [], []
        pos_total = (score_col['label'] == 1.0).sum()
        neg_total = (score_col['label'] == 0.0).sum()

        for th in thresholds:
            pred = (score_col['score'] >= th).astype(int)
            tp_  = ((score_col['label'] == 1.0) & (pred == 1)).sum()
            fp_  = ((score_col['label'] == 0.0) & (pred == 1)).sum()
            tprs.append(tp_ / pos_total if pos_total else 0)
            fprs.append(fp_ / neg_total if neg_total else 0)

        ax.plot(fprs, tprs, color=roc_colors[i], lw=2,
                label=f"{mname.replace('Classifier','').replace('Regression','Reg')} (AUC={auc:.3f})")
    except Exception:
        ax.plot([0,1],[0,1], color=roc_colors[i], lw=1, linestyle=':', label=f'{mname} (no prob)')

ax.plot([0,1],[0,1], 'k--', lw=1, label='Random (AUC=0.5)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — All Models', fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.set_xlim(0,1); ax.set_ylim(0,1.02)
plt.tight_layout()
savefig('roc_curve.png')
plt.show()

## 9. Özet Tablosu

In [ ]:
# Save final metrics CSV
metrics_df.to_csv(REPORTS_DIR / 'ml_model_metrics.csv', index=False)

best = metrics_df.loc[metrics_df['auc_roc'].idxmax()]

lines = [
    '| Model | Accuracy | F1 | AUC-ROC | Recall (arrested) |',
    '|-------|----------|----|---------|-------------------|',
]
for _, r in metrics_df.iterrows():
    marker = ' 🏆' if r['model'] == best_model_name else ''
    lines.append(
        f"| {r['model']}{marker} | {r['accuracy']:.4f} | {r['f1']:.4f} "
        f"| {r['auc_roc']:.4f} | {r['recall_arrested']:.4f} |"
    )
lines += [
    '',
    f'**Best model:** {best_model_name} — AUC-ROC = {best_auc:.4f}',
    f'**MLflow runs:** `{MLFLOW_URI}`  →  open http://localhost:5001',
    f'**Charts saved to:** `{FIGURES_DIR}`',
]
display(Markdown('\n'.join(lines)))

spark.stop()
print('Done.')